In [ ]:
import sys
sys.path.append(
    r"D:\Papers\4) Finished Articles\6. MWSN - DCHPC\DCHPC\ModelClasses")
from simulation import Simulation

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
from tqdm import tqdm

from simulation import Simulation

### Fixed simulation configuration

In [ ]:
OUTPUT_DIR = r"D:\Papers\4) Finished Articles\6. MWSN - DCHPC\DCHPC\AParameters_tuning"

In [ ]:
BASE_CONFIG = dict(
    area_size=(100, 100),
    n_nodes=100,
    rounds=60000,
    init_energy=0.5,
    comm_range=50.0,
    sink_mode="adaptive",
    routing_mode="multi-hop",
    localization_mode="random",
    seed=42,
    head_selection_strategy="optimizer"
)

In [ ]:
# run single sim and extract metrics
def run_sim_get_metrics(extra_config):
    try:
        sim = Simulation(**BASE_CONFIG, **extra_config)
        metrics = sim.run()
        return {
            'LND': metrics.get('LND', 0),
            'PDR': metrics.get('PDR', 0),
            'Avg_E2E_Delay_Sec': metrics.get('Avg_E2E_Delay_Sec', 0)
        }
    except Exception as e:
        print(f"Failed: {e}")
        return {'LND': 0, 'PDR': 0, 'Avg_E2E_Delay_Sec': float('inf')}

### Routing Weights Tuning

In [ ]:
# Weight grid: we fix sum = 1.0
# Focus on reasonable ranges based on intuition
distance_vals = [0.4, 0.5, 0.6]
energy_vals = [0.2, 0.3, 0.4]
load_vals = [0.05, 0.1, 0.15]
trust_vals = [0.05, 0.1, 0.15]

In [ ]:
def valid_combo(wd, we, wl, wt):
    return abs(wd + we + wl + wt - 1.0) < 1e-6


# Generate valid combinations
combinations = []
for wd, we, wl, wt in itertools.product(distance_vals, energy_vals, load_vals, trust_vals):
    if valid_combo(wd, we, wl, wt):
        combinations.append((wd, we, wl, wt))
print(f"Testing {len(combinations)} routing weight combinations.")

In [ ]:
routing_results = []
for weights in tqdm(combinations, desc="Routing"):
    wd, we, wl, wt = weights
    metrics = run_sim_get_metrics({
        'weight_distance': wd,
        'weight_energy': we,
        'weight_load': wl,
        'weight_trust': wt
    })
    routing_results.append({
        'param_group': 'routing',
        'weight_distance': wd,
        'weight_energy': we,
        'weight_load': wl,
        'weight_trust': wt,
        **metrics
    })

In [ ]:
pd.DataFrame(routing_results).to_csv(
    os.path.join(OUTPUT_DIR, "routing.csv"), index=False)

In [ ]:
df_r = pd.DataFrame(routing_results)
best_r = df_r.loc[df_r['LND'].idxmax()]
print(f"Routing: d={best_r['weight_distance']}, e={best_r['weight_energy']}, "
      f"l={best_r['weight_load']}, t={best_r['weight_trust']} → LND={best_r['LND']:.0f}")

In [ ]:
df_r.head(5).sort_values('LND', ascending=False)

### Reclustering Parameters Tuning

In [ ]:
reclust_defaults = {
    'recluster_period': 50,
    'energy_threshold': 0.3,
    'load_threshold': 10,
    'sink_move_threshold': 20.0
}
reclust_tests = {
    'recluster_period': [10, 20, 30, 50, 75, 100],
    'energy_threshold': [0.1, 0.2, 0.3, 0.4, 0.5],
    'load_threshold': [5, 10, 15, 20, 25],
    'sink_move_threshold': [10, 15, 20, 25, 30]
}

In [ ]:
reclust_results = []
for param, values in reclust_tests.items():
    for val in tqdm(values, desc=f"Reclust: {param}"):
        config = {k: v for k, v in reclust_defaults.items() if k != param}
        config[param] = val
        metrics = run_sim_get_metrics(config)
        reclust_results.append({
            'param_group': 'reclustering',
            'tested_param': param,
            'tested_value': val,
            **metrics
        })

In [ ]:
pd.DataFrame(reclust_results).to_csv(os.path.join(
    OUTPUT_DIR, "reclustering_params.csv"), index=False)

In [ ]:
test = []
df_rc = pd.DataFrame(reclust_results)
for param in reclust_tests.keys():
    subset = df_rc[df_rc['tested_param'] == param]
    test.append(subset.sort_values(by='LND', ascending=False))
    best = subset.loc[subset['LND'].idxmax()]
    print(f"Reclust {param}: {best['tested_value']} → LND={best['LND']:.0f}")

In [ ]:
test[0].head(3)

In [ ]:
test[1].head(3)

In [ ]:
test[2].head(3)

In [ ]:
test[3].head(3)

### Gravitational Optimizer Parameters Tuning

In [ ]:
go_tests = {
    'go_iterations': [5, 10, 15, 20, 25],
    'population_size': [5, 10, 15, 20],
    'alpha': [0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
    'G0': [50, 100, 150, 200]
    # beta = 1 - alpha, so not tested independently
}

In [ ]:
go_results = []
for param, values in go_tests.items():
    for val in tqdm(values, desc=f"GO: {param}"):
        if param == 'alpha':
            # Auto-set beta = 1 - alpha
            metrics = run_sim_get_metrics({'alpha': val, 'beta': 1.0 - val})
        else:
            metrics = run_sim_get_metrics({param: val})
        go_results.append({
            'param_group': 'optimizer',
            'tested_param': param,
            'tested_value': val,
            **metrics
        })

In [ ]:
pd.DataFrame(go_results).to_csv(os.path.join(
    OUTPUT_DIR, "gravitational_params.csv"), index=False)

In [ ]:
test = []
df_rc = pd.DataFrame(go_results)
for param in go_tests.keys():
    subset = df_rc[df_rc['tested_param'] == param]
    test.append(subset.sort_values(by='LND', ascending=False))
    best = subset.loc[subset['LND'].idxmax()]
    print(f"OGS {param}: {best['tested_value']} → LND={best['LND']:.0f}")

In [ ]:
test[0].head(3)

In [ ]:
test[1].head(3)

In [ ]:
test[2].head(3)

In [ ]:
test[3].head(3)

### Mobile Sink Parameters Tuning

In [ ]:
sink_tests = {
    'visit_period': [2, 3, 5, 7, 10],
    'energy_weight': [0.4, 0.5, 0.6, 0.7, 0.8],
    # distance_weight = 1 - energy_weight, so not tested independently
}

In [ ]:
sink_results = []
for param, values in sink_tests.items():
    for val in tqdm(values, desc=f"Sink: {param}"):
        if param == 'energy_weight':
            # Auto-set distance_weight = 1 - energy_weight
            metrics = run_sim_get_metrics(
                {'distance_weight': val, 'energy_weight': 1.0 - val})
        else:
            metrics = run_sim_get_metrics({param: val})
        sink_results.append({
            'param_group': 'sink',
            'tested_param': param,
            'tested_value': val,
            **metrics
        })

In [ ]:
pd.DataFrame(sink_results).to_csv(os.path.join(
    OUTPUT_DIR, "mobilesink_params.csv"), index=False)

In [ ]:
test = []
df_rc = pd.DataFrame(sink_results)
for param in sink_tests.keys():
    subset = df_rc[df_rc['tested_param'] == param]
    test.append(subset.sort_values(by='LND', ascending=False))
    best = subset.loc[subset['LND'].idxmax()]
    print(f"OGS {param}: {best['tested_value']} → LND={best['LND']:.0f}")

In [ ]:
test[0].head(3)

In [ ]:
test[1].head(3)

## Systematic ablation or robustness study to gain stable and not highly parameter-dependent performance

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, spearmanr
from tqdm import tqdm

In [ ]:
# 1. CONFIGURATION & DEFINITIONS
OUT_DIR = "reviewer_response_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, "plots"), exist_ok=True)

# METRICS & RUN SETTINGS
METRICS_OF_INTEREST = [
    "FND", "HND", "LND", "PDR", "Avg_E2E_Delay_Sec",
    "Avg_E2E_Delay_Rounds", "TotalGenerated", "TotalDelivered", "EE_Js", "Routing_Overhead_Bytes"
]

# Standard research settings
N_SEEDS = [
    1, 3, 4, 5, 7, 10, 11, 13, 17, 18,
    19, 23, 29, 31, 37, 41, 43, 47, 53, 59,
    61, 67, 71, 73, 79, 83, 89, 97, 101, 103,
    107, 109, 113, 127, 131, 137, 139, 149, 151, 157,
    163, 167, 173, 179, 181, 191, 193, 197, 199, 211,
    223, 227, 229, 233, 239, 241, 251, 257, 263, 269
]         # Number of statistical runs per config
GLOBAL_SAMPLES = 200   # Increased for better global coverage
SEEDS_PER_SAMPLE = 3   # Low seeds for global exploration

# BASE HYPERPARAMETERS (The "Proposed Method")
BASE_CONFIG = {
    "area_size": (100, 100),
    "n_nodes": 100,
    "rounds": 60000,
    "init_energy": 0.5,
    "comm_range": 50.0,
    "sink_mode": "eeosp",
    "routing_mode": "multi-hop",
    "localization_mode": "DRL",
    "head_selection_strategy": "optimizer",
    "include_ack_energy": False,
    "num_sinks": 1,
    "enable_heterogeneity": False,
    "hetero_mode": "two_tier",
    "variable_packet_size": False,

    # 1. Routing Fitness
    "weight_distance": 0.5,
    "weight_energy": 0.3,
    "weight_load": 0.1,
    "weight_trust": 0.1,

    # 2. OGSA Optimization
    "go_iterations": 15,
    "population_size": 10,
    "G0": 50.0,
    "alpha": 0.6, "beta": 0.4,

    # 3. Mobile Sink (EEOSP)
    "energy_weight": 0.4,
    "distance_weight": 0.6,
    "visit_period": 5,

    # 4. Node Placement (Global Rewards)
    "edge_threshold": 0.4,
    "tune_edge_iterations": 20,
    "reward_coverage_weight": 0.35,
    "reward_edge_coverage_weight": 0.30,
    "reward_connectivity_score_weight": 0.20,
    "reward_uniformity_weight": 0.15,

    # 5. Node Placement (Local Rewards)
    "local_reward_coverage_score_weight": 0.30,
    "local_reward_connectivity_score_weight": 0.25,
    "local_reward_boundary_score_weight": 0.30,
    "local_reward_overlap_penalty_weight": 0.15,
    "q_alpha": 0.2, "q_gamma": 0.8, "q_epsilon": 0.9,

    # Clustering
    "k_min": 8, "k_max": 20,
    "recluster_period": 75,
    "energy_threshold": 0.1, "load_threshold": 10, "sink_move_threshold": 20.0
}


# WEIGHT GROUPS (For Smart Normalization)
# This ensures that when we change one weight, the others adjust
# to keep the sum == 1.0, preserving mathematical validity.
WEIGHT_GROUPS = {
    "Routing": ["weight_distance", "weight_energy", "weight_load", "weight_trust"],
    "OGSA_Coeffs": ["alpha", "beta"],
    "Mobile_Sink": ["energy_weight", "distance_weight"],
    "Global_Rewards": [
        "reward_coverage_weight", "reward_edge_coverage_weight",
        "reward_connectivity_score_weight", "reward_uniformity_weight"
    ],
    "Local_Rewards": [
        "local_reward_coverage_score_weight",
        "local_reward_connectivity_score_weight",
        "local_reward_boundary_score_weight",
        "local_reward_overlap_penalty_weight"
    ]
}

In [ ]:
# 2. DEFINING THE STUDY SCENARIOS
# A. OAT (One-At-A-Time) SENSITIVITY RANGES
# We test a Low, Base, and High value for every weight.
PARAM_RANGES = {}

# Helper to generate ranges around the base
def make_range(base_val):
    # Try -50%, Base, +50% (clipped 0.0 to 1.0)
    low = round(max(0.05, base_val * 0.5), 2)
    high = round(min(0.95, base_val * 1.5), 2)
    # Ensure distinct values
    vals = sorted(list(set([low, base_val, high])))
    # If base is 0.5, we might want stronger extremes for the paper
    if len(vals) < 3:
        vals = [0.2, 0.5, 0.8]
    return vals

# 1. Routing
for k in WEIGHT_GROUPS["Routing"]:
    PARAM_RANGES[k] = [0.1, 0.3, 0.5, 0.7, 0.9]  # Standard sweep

# 2. OGSA
PARAM_RANGES["alpha"] = [0.2, 0.4, 0.6, 0.8]  # Beta will auto-adjust
PARAM_RANGES["G0"] = [25.0, 50.0, 100.0]

# 3. Mobile Sink
PARAM_RANGES["energy_weight"] = [
    0.2, 0.4, 0.6, 0.8]  # Distance will auto-adjust

# 4. Global Rewards
for k in WEIGHT_GROUPS["Global_Rewards"]:
    PARAM_RANGES[k] = make_range(BASE_CONFIG[k])

# 5. Local Rewards
for k in WEIGHT_GROUPS["Local_Rewards"]:
    PARAM_RANGES[k] = make_range(BASE_CONFIG[k])

In [ ]:
# B. ABLATION SCENARIOS (Turning features OFF)
ABLATION_SCENARIOS = {
    #  1. Routing Fitness Ablation
    "Routing_No_Energy": {"weight_energy": 0.0},
    "Routing_No_Trust": {"weight_trust": 0.0},
    "Routing_No_Load": {"weight_load": 0.0},
    "Routing_No_Distance": {"weight_distance": 0.0},

    #  2. OGSA (Optimization) Ablation
    "OGSA_No_Distance": {"alpha": 0.0},
    "OGSA_No_Energy": {"beta": 0.0},

    # #  3. Global Reward (Placement) Ablation
    # "Placement_No_Edge_Reward": {"reward_edge_coverage_weight": 0.0},
    # "Placement_No_Connectivity": {"reward_connectivity_score_weight": 0.0},
    # "Placement_No_Coverage": {"reward_coverage_weight": 0.0},
    # "Placement_No_Uniformity": {"reward_uniformity_weight": 0.0},

    # #  4. Local Reward (D-RL) Ablation
    # "Local_No_Overlap_Penalty": {"local_reward_overlap_penalty_weight": 0.0},
    # "Local_No_Boundary_Awareness": {"local_reward_boundary_score_weight": 0.0},
    # "Local_No_Coverage": {"local_reward_coverage_score_weight": 0.0},
    # "Local_No_Connectivity": {"local_reward_connectivity_score_weight": 0.0},

    #  5. EEOSP (Optimization) Ablation
    "EEOSP_No_Distance": {"distance_weight": 0.0},
    "EEOSP_No_Energy": {"energy_weight": 0.0},
}

In [ ]:
# 3. HELPER FUNCTIONS
def safe_extract_metric(metrics, detailed, key):
    if metrics and key in metrics:
        return metrics[key]
    aliases = {
        "Avg_E2E_Delay_Sec": ["avg_delay", "E2E_Delay"],
        "Routing_Overhead_Bytes": ["routing_overhead", "overhead"],
        "EE_Js": ["EnergyEfficiency", "EE"]
    }
    if key in aliases:
        for alias in aliases[key]:
            if metrics and alias in metrics:
                return metrics[alias]
    if detailed:
        if key in detailed and len(detailed[key]) > 0:
            return detailed[key][-1]
        if key in aliases:
            for alias in aliases[key]:
                if alias in detailed and len(detailed[alias]) > 0:
                    return detailed[alias][-1]
    return np.nan

In [ ]:
def get_normalized_config(base_cfg, overrides):
    """
    Apply overrides and strictly enforce Sum=1 for weight groups.
    If 'weight_distance' is changed, remaining weight is distributed
    among sibling weights proportional to their original values.
    """
    new_cfg = base_cfg.copy()
    new_cfg.update(overrides)

    # Check every group for normalization validity
    for group_name, keys in WEIGHT_GROUPS.items():
        # Check if any key in this group was modified
        if any(k in overrides for k in keys):
            # Identify which keys were manually set (locked)
            locked_keys = [k for k in keys if k in overrides]
            floating_keys = [k for k in keys if k not in overrides]

            current_sum_locked = sum(new_cfg[k] for k in locked_keys)

            if current_sum_locked > 1.0:
                # If single param > 1 (unlikely) or multiple locked exceed 1, normalize locked only
                # (This is a simplified handling, assuming usually 1 locked in OAT)
                factor = 1.0 / current_sum_locked
                for k in locked_keys:
                    new_cfg[k] *= factor
                for k in floating_keys:
                    new_cfg[k] = 0.0

            elif len(floating_keys) > 0:
                # Distribute remaining (1 - locked) among floating
                remaining_budget = 1.0 - current_sum_locked

                # Get base proportions of floating keys
                base_sum_floating = sum(base_cfg[k] for k in floating_keys)

                if base_sum_floating == 0:
                    # If they were 0, distribute equally
                    for k in floating_keys:
                        new_cfg[k] = remaining_budget / len(floating_keys)
                else:
                    # Distribute proportionally
                    for k in floating_keys:
                        ratio = base_cfg[k] / base_sum_floating
                        new_cfg[k] = remaining_budget * ratio

            # Final sanity check rounding to avoid 0.999999
            total = sum(new_cfg[k] for k in keys)
            if abs(total - 1.0) > 1e-5:
                # Force normalize last key
                new_cfg[keys[-1]] += (1.0 - total)
    print(new_cfg)
    return new_cfg

In [ ]:
def run_simulation_wrapper(config):
    """
    mockup or import your actual Simulation class here.
    Ensure you import: from my_simulation_package import Simulation
    """
    try:
        sim = Simulation(**config)
        metrics, detailed = sim.run()

        return metrics, detailed
    except Exception as e:
        print(f"Sim Error: {e}")
        return None, None

In [ ]:
def run_batch(base_cfg, overrides_list, seeds):
    results = []
    for params in overrides_list:
        # Apply normalization logic
        valid_cfg = get_normalized_config(base_cfg, params)

        for seed in seeds:
            run_cfg = valid_cfg.copy()
            run_cfg['seed'] = seed

            # Run
            metrics, detailed = run_simulation_wrapper(run_cfg)

            if metrics:
                row = {"seed": seed}
                row.update(params)  # Log what we changed
                for m in METRICS_OF_INTEREST:
                    row[m] = safe_extract_metric(metrics, detailed, m)
                results.append(row)
    return pd.DataFrame(results)

In [ ]:
# 4. EXECUTION ROUTINES
def run_systematic_ablation():
    print("\n 1. Systematic Ablation Study ")
    seeds = N_SEEDS

    # Prepare list of configs
    configs_to_run = []

    # 1. Baseline
    print(" 1. Baseline")
    BASE_CONFIG['localization_mode'] = 'random'
    df_base = run_batch(BASE_CONFIG, [{}], seeds)
    df_base['Scenario'] = 'Baseline_Method'

    print(" 2. Proposed Method")
    BASE_CONFIG['localization_mode'] = 'DRL'
    df_rl = run_batch(BASE_CONFIG, [{}], seeds)
    df_rl['Scenario'] = 'Proposed_Method'

    dfs = [df_base, df_rl]

    for name, overrides in ABLATION_SCENARIOS.items():
        print('\n')
        print(f"Running Ablation: {name}")
        df = run_batch(BASE_CONFIG, [overrides], seeds)
        df['Scenario'] = name
        dfs.append(df)

    full_df = pd.concat(dfs, ignore_index=True)
    full_df.to_csv(os.path.join(OUT_DIR, "ablation_results.csv"), index=False)

    # Plotting
    for metric in METRICS_OF_INTEREST:
        plt.figure(figsize=(10, 6))
        sns.barplot(data=full_df, x="Scenario",
                    y=metric, ci=95, palette="RdBu_r")
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Ablation Analysis: {metric}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "plots", f"Ablation_{metric}.png"))
        plt.close()

    return full_df


def run_oat_sensitivity():
    print("\n 2. OAT Sensitivity Analysis (with Normalization) ")
    seeds = N_SEEDS

    summary_data = []

    # Get Baseline Means for comparison
    df = pd.read_csv(os.path.join(OUT_DIR, "ablation_results.csv"))
    df_base = df[df['Scenario'] == 'Baseline_Method']
    df_rl = df[df['Scenario'] == 'Proposed_Method']
    df_means = df_base[METRICS_OF_INTEREST].mean()
    rl_means = df_rl[METRICS_OF_INTEREST].mean()

    for param, values in PARAM_RANGES.items():
        print(f"Testing {param} values: {values}")

        # create list of override dicts
        overrides_list = [{param: v} for v in values]

        df = run_batch(BASE_CONFIG, overrides_list, seeds)

        # Calculate stats
        grouped = df.groupby(param)[METRICS_OF_INTEREST].mean()

        for v in values:
            if v in grouped.index:
                row = {"Parameter": param, "Value": v}
                for m in METRICS_OF_INTEREST:
                    val = grouped.loc[v, m]
                    row[f"{m}_Mean"] = val
                    # Sensitivity Score: |(New - Base) / Base|
                    if rl_means[m] != 0:
                        row[f"{m}_Sens%"] = (
                            (val - rl_means[m]) / rl_means[m]) * 100
                summary_data.append(row)

        # Spider/Line Plot per parameter
        plt.figure(figsize=(8, 5))
        # Normalize to % change for plotting
        plot_df = df.groupby(param)[METRICS_OF_INTEREST].mean()
        for col in plot_df.columns:
            plot_df[col] = (plot_df[col] / rl_means[col]) * 100

        sns.lineplot(data=plot_df, markers=True, dashes=False)
        plt.title(f"Sensitivity of Metrics to {param}")
        plt.ylabel("% of Baseline Performance")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, "plots", f"OAT_{param}.png"))
        plt.close()

    pd.DataFrame(summary_data).to_csv(os.path.join(
        OUT_DIR, "oat_summary.csv"), index=False)


def run_global_robustness():
    print("\n 3. Global Monte Carlo Robustness ")
    # Generate valid random configurations (Dirichlet-like)

    random_configs = []
    for _ in range(GLOBAL_SAMPLES):
        cfg = {}

        # Helper to get random normalized weights
        def get_rand_weights(keys):
            raw = np.random.rand(len(keys))
            return raw / raw.sum()

        # 1. Routing
        r_w = get_rand_weights(WEIGHT_GROUPS["Routing"])
        for i, k in enumerate(WEIGHT_GROUPS["Routing"]):
            cfg[k] = r_w[i]

        # 2. Global Rewards
        g_w = get_rand_weights(WEIGHT_GROUPS["Global_Rewards"])
        for i, k in enumerate(WEIGHT_GROUPS["Global_Rewards"]):
            cfg[k] = g_w[i]

        # 3. Local Rewards
        l_w = get_rand_weights(WEIGHT_GROUPS["Local_Rewards"])
        for i, k in enumerate(WEIGHT_GROUPS["Local_Rewards"]):
            cfg[k] = l_w[i]

        # 4. Misc
        cfg["alpha"] = np.random.uniform(0.1, 0.9)
        cfg["beta"] = 1.0 - cfg["alpha"]
        cfg["k_min"] = np.random.randint(5, 15)

        random_configs.append(cfg)

    # Run
    full_results = run_batch(
        BASE_CONFIG, random_configs, range(SEEDS_PER_SAMPLE))

    # Correlation Analysis
    # Flatten config cols
    # (Since we passed full dicts, run_batch might have them as cols if keys match)
    # If not, we merge.

    # Calculate Spearman Correlation
    # Identify parameter columns in result
    param_cols = [
        k for k in full_results.columns if k in BASE_CONFIG.keys() and k != 'seed']

    corr_data = []
    for metric in METRICS_OF_INTEREST:
        for param in param_cols:
            if full_results[param].nunique() > 1:
                corr, _ = spearmanr(full_results[param], full_results[metric])
                corr_data.append(
                    {"Parameter": param, "Metric": metric, "Correlation": corr})

    corr_df = pd.DataFrame(corr_data)

    # Plot Heatmap
    heatmap_data = corr_df.pivot(
        index="Parameter", columns="Metric", values="Correlation")
    plt.figure(figsize=(12, 10))
    sns.heatmap(heatmap_data, annot=True, cmap="coolwarm",
                center=0, vmin=-1, vmax=1)
    plt.title(
        "Global Sensitivity: Spearman Correlation\n(Values near 0 imply robustness)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "plots", "Global_Sensitivity_Map.png"))
    plt.close()

    print("Robustness check complete.")

## Ablation (Rationale for components)

In [ ]:
# 1. Ablation (Rationale for components)
run_systematic_ablation()

## OAT (Rationale for specific weights)

In [ ]:
N_SEEDS = [1, 3, 4, 18, 29, 31, 41, 53, 61, 67,
           73, 83, 89, 97, 107, 109, 137, 139, 163, 173,
           179, 181, 191, 197, 227, 229, 233, 239, 241, 251]

In [ ]:
# 2. OAT (Rationale for specific weights)
run_oat_sensitivity()

## Global (Proof of stability)

In [ ]:
# 3. Global (Proof of stability)
run_global_robustness()

## wilcoxon

In [ ]:
from scipy.stats import wilcoxon

baseline_vals = df_base["PDR"].values
ablation_vals = df_ablation["PDR"].values
stat, p = wilcoxon(baseline_vals, ablation_vals)
print(f"PDR difference significant? p={p:.4f} (α=0.05)")